In [ ]:
import sisl
import matplotlib.pyplot as plt
import numpy as np
from TimedependentTransport import TD_Transport, AdaptiveRK4,hbar
from numba import njit
from funcs import diff_central, calc_current
from ase import Atoms
from ase.build import molecule
from ase.visualize import view
import sisl

### In this block we define the geometry. All standard sisl stuff.

In [ ]:
#Define sampling of the lead Gammas
eps  = (1e-6 + 1j  * 1e-2)
line = np.linspace(-8, 8, 152) + eps
# line = np.array([-8, -4, -2.5, -1.0,  1.0, 2.0 ,3.0, 4.0, 5.0, 6.0, 7.0, 8.0]) + eps
line = np.vstack((line,line))
# make a molecule
BDA = molecule('BDA')
BDA.cell = np.diag(1.44 * np.ones(3))

#BDA.edit() COMMENT THIS IN IF YOU WANT TO MODIFY THE DEVICE WITH ASE

BDA = sisl.Geometry.fromASE(BDA)
# make leads
move_up = 5.0
move_dw = -14.8
L  = sisl.geom.sc(1.44, 'C').tile(8,1)
LU = L.move([0,move_up,0]).add_vacuum(10,0).add_vacuum(10,2)
LD = L.move([0,move_dw,0]).add_vacuum(10,0).add_vacuum(10,2)
dev = BDA.add(LU).add(LD)
dev = dev.add_vacuum(20,0).add_vacuum(dev.xyz[:,1].max() - dev.xyz[:,1].min(),1).add_vacuum(20,2)
dev = dev.move(dev.cell[1,:]/2)
LU  = sisl.geom.sc(1.44, 'C').tile(4,1).move([0,move_up,0]).add_vacuum(10,0).add_vacuum(10,2)
LD  = sisl.geom.sc(1.44, 'C').tile(4,1).move([0,move_dw,0]).add_vacuum(10,0).add_vacuum(10,2)
LU  = LU.move(LU.cell[1,:])
LD  = LD.move(-LD.cell[1,:]/2 + np.array([0, 2 * 1.44, 0]))
LU  = LU.move (dev.cell[1,:]/2)
LD  = LD.move (dev.cell[1,:]/2)

plt.show()
sisl.plot(dev); plt.axis('equal')

In [ ]:
Test = TD_Transport([LD,LU], dev, kT_i = [0.025, 0.025])
Test.Make_Contour(line, 10, pole_mode = 'JieHu2011')
plt.show()
Test.Electrodes( semi_infs = ['-a2', '+a2'] , kp=[[1,50,1],[1,50,1]])
Test.make_device()
plt.show()
Test.Device.Visualise()
plt.show()

In [ ]:
Test.run_electrodes()
Test.run_device()

In [ ]:
# Get the pz part of the system. Hydrogens should only affect the Carbon onsite potential

sub_indices = []
TBT = sisl.get_sile(Test.Device.dir + '/siesta.TBT.nc')
count = 0
atoms = [TBT.geom.atoms[i] for i in range(TBT.geom.na) if i in TBT.a_dev]
for ia, a in enumerate(atoms):
    for io in range(TBT.geom.orbitals[ia]):
        if io == 2:
            sub_indices += [count]
        count+=1

Test.read_data(fact = 2.0, NumL=25, #Fallback_W= 20,
                    fit_mode = 'all', use_analytical_jac = True, 
                    maxiter = 10 ** 6,min_method = 'L-BFGS-B', 
                    ebounds = (line.min()-1 , line.max()+1 ), 
                    wbounds = (0.01, 5.0), 
                    tol = -1.0, options = {'disp':True, 'maxiter': 10**6, 'gtol': 0.1},
                    sub_orbital = sub_indices # Not really using sub_orbital from sisl, but the idea is the same I guess
                    )

plt.show()
Test.get_propagation_quantities(); Test.get_dense_matrices()

Vi = np.linspace(-.4,.4,20); idx = np.argsort(np.abs(Vi)); Vi = Vi[idx]
Test.run_device_non_eq(Vi)
h,v = Test.neq_Hs()

### Here we read in the data from transiesta and fit the $\Gamma$'s using the L-BFGS-B method. Takes a couple of minutes

### Inspect the fitting of the $\Gamma$'s

In [ ]:
I,J,i,j  = 0,0,1,1 #I,J Blocks from tbtrans, i,j are the subindices in these blocks
Test.Inspect_Lorentzian_fit(1,I,J,i,j, Emin = -10,Emax= 10)

### Inspect the Transmission function of the pz part:

In [ ]:
Test.Inspect_Transmission(0,1)

In [ ]:
#Here we do the calculation of the quantities that goes into the equations of motion:
Test.get_propagation_quantities()
Test.get_dense_matrices()
sig = Test.sigma
psi = Test.Psi_vec
omega = Test.omegas
no_d = sig.shape[2]

### Plot of the eigenvalues of the $\mathbf{\Lambda}_{\alpha l}$ matrices: 
Evidently (from the plot below) there are a lot of values that are zero to machine precision. In the equation of motion (Eq. 21 in Croy 2016) for $\Omega_{\alpha x c , \alpha' x' c'}$ these are important:


$i\hbar\frac{\mathrm{d}\Omega_{\alpha x c , \alpha' x' c'}}{\mathrm{d} t} = [\chi^-_{\alpha' x' c' } + \Delta_\alpha ' (t)  -\chi^+_{\alpha x c } - \Delta_\alpha(t) ]\Omega_{\alpha x c , \alpha' x' c'} + i\hbar (\Lambda_{\alpha'x'c'}^{>,-} - \Lambda_{\alpha'x'c'}^{<,-})\mathbf{\xi}^\dagger\mathbf{\Psi} + i\hbar(\Lambda_{\alpha x c}^{>,+} - \Lambda_{\alpha x c}^{<,+})\mathbf{\Psi}^\dagger\mathbf{\xi} $ .

This is because when all four of the $\Lambda$'s in the two latter terms are zero, this particular $\Omega$ stays zero identically, as there is no driving term to get it going. These particular elements of $\Omega$ need therefore not be stored. This is in addition to the elements of $\Omega$ containing labels from the same fermi function.

These zeros comes from the original $\Gamma$'s from TBtrans being only located in one corner of its matrix. Doing the Löwdin transformation preserves the eigenvalues of an hermitian matrix, which results in us getting a lot of zeros even efter we have smeared out the orbitals with the Löwdin transform. See the figure below $\downarrow$

In [ ]:
_ = plt.plot(np.log10(Test.Gl_eig[0,0]).real)

### Where is the stuff stored
We now have all the quantities we need to initialize and propagate the equations of motion for the density matrix and so on. $\Lambda^{</>, +/-}_{\alpha x c}$ are located in the variables "Test.GG_P", "Test.GG_L", "Test.GL_P" and "Test.GL_M". Here the values are stored as an array with shape 

(#kpts, #leads, #auxillary modes, #device orbitals)

The Lorentzian and Fermi poles are stored in arrays with the same shape in the variables "Test.Xpp" and "Test.Xpm".


The original eigenvalues (which are now superflous, only needed for the calculation of $\Lambda^{</>, +/-}$) and the corresponding eigenvectors(which are still needed) are stored in "Test.Gl_eig", "Test.Gp_eig", "Test.Gl_vec" (of which the inverse is just the hermitian conjugate) and Test.Gp_vec (of which its inverse is stored in "Test.Inv_Gp_vec")


In [ ]:

print('GG_P shape: ', Test.GG_P.shape)
print('GG_M shape: ', Test.GG_M.shape)
print('GL_P shape: ', Test.GL_P.shape)
print('GL_M shape: ', Test.GL_M.shape)
print('Gl_eig shape: ', Test.Gl_eig.shape)
print('Gl_vec shape: ', Test.Gl_vec.shape)
print('below they contains 2 * #num_fermi_poles (Upper AND lower poles) in the third index')
print('Gp_eig shape: ', Test.Gp_eig.shape)
print('Gp_vec shape: ', Test.Gp_vec.shape)
print('Inv_Gp_vec shape: ', Test.Inv_Gp_vec.shape)
print('21 + 24/2 gives the length of the third index of the top four arrays')





## We now go ahead with the actual propagation scheme

In [ ]:
from Peter_Uhd import air_photonics_pulse

V = 1.0
@njit
def Bias(t, tp, ts):
    return V * (np.tanh(t / ts) - np.tanh((t - tp) / ts)) / 2
@njit
def fd(x, mu, s):
    return 1 / (1 + np.exp((x - mu) / s))
@njit
def delta(t, a):
    if a == 0:
        return air_photonics_pulse(t)  #Bias(t, 20, 1)/2
    elif a == 1:
        return -air_photonics_pulse(t)#- Bias(t, 20, 1)/2
@njit
def zero_bias(t, a):
    return 0.0
@njit
def zero_dH(t, sigma):
    A = np.zeros((1, no_d, no_d), dtype=np.complex128)
    return A

@njit
def dH(t, sigma):
    # We can put in Hartree correction here like in 
    # "Insights into the Charge Separation Dynamics in Photoexcited
    # Molecular Junctions"
    # 
    # For now we just put a sine somewhere
    A = np.zeros((1, no_d, no_d), dtype=np.complex128)
    #A[0,5,5] = np.sin(t) * V/4 * Bias(t, 20, 1)
    return A

f   = Test.make_f_general(parallel = True, fastmath = True)
xi  = Test.xi
Ixi = Test.Ixi

t1, data1 = AdaptiveRK4(
                        f,
                        sig, psi, omega,
                        1e-5, -15.0, 10.0,
                        zero_dH,
                        zero_bias,
                        Test.Ixi,
                        0.01,
                        fixed_mode=False,
                        name="Multithread_RK4",
)

sig          = data1['last sigma']
omega        = data1['last omega']
psi          = data1['last psi']

t2, data2 = AdaptiveRK4(
                        f,
                        sig, psi, omega,
                        1e-6, -5.0, 500.0,
                        dH,
                        delta,
                        Test.Ixi,
                        0.01,
                        fixed_mode=False,
                        name="Multithread_RK4",
)

In [ ]:
dNdt = diff_central(t2, np.array([np.trace(data2['density matrix'][i]) for i in range(len(t2))]))
JL   = data2['current left']
JR   = data2['current right']

### Lets plot the left and right currents!
First we do the nonbiased case and then the biased case

In [ ]:
plt.plot(t1, data1['current left'], label = r'$J_L$')
plt.plot(t1, data1['current right'], label = r'$J_R$')
plt.xlabel('time [fs]',size = 18)
plt.ylabel(r'current [$\frac{e}{fs}$]',size = 18)
plt.title('Equilibrium')
plt.legend()

In [ ]:
plt.plot(t2, JL, label = r'$J_L$')
plt.plot(t2, JR, label = r'$J_R$')
plt.xlabel('time [fs]',size = 18)
plt.ylabel(r'current [$\frac{e}{fs}$]',size = 18)
plt.title('With biaspulse, using the equilibrium state as starting point')
plt.legend()

In [ ]:
plt.plot(t2[1:-1], dNdt, label = r'$0.95*\frac{dN}{dt}$')
plt.plot(t2, JL+ JR, label = r'$J_L + J_R$')
plt.legend()
plt.xlabel('time [fs]',size = 18)
plt.ylabel(r'current [$\frac{e}{fs}$]',size = 18)
plt.title('Check for charge conservation')


In [ ]:
from scipy.integrate import simpson
print(simpson(t2, JL + JR))

In [ ]:
plt.plot(t2, data2['density matrix'][:,0,0], label = r'$J_L + J_R$')